# 第2章 Python 数据科学工具栈 —— 配套代码

对应正文 `book/part1/02-python-toolkit.md`。

> **运行前请先生成内置数据：**
> ```bash
> uv run python scripts/make_sample_data.py
> ```

本 notebook 涵盖：
- NumPy ndarray / dtype / axis / 向量化性能对比 / 广播 / 矩阵乘
- Pandas loc/iloc/布尔索引 / DatetimeIndex / 缺失值 / groupby / merge / resample / rolling+ewm
- Parquet 往返一致性验证
- matplotlib 含中文标注的可视化
- 习题参考解答

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fds import load_sample_prices, daily_returns, set_chinese_font

# 设置中文字体（全局生效）
set_chinese_font()

print(f"NumPy  版本：{np.__version__}")
print(f"Pandas 版本：{pd.__version__}")

## §1  NumPy：ndarray、dtype 与 axis

NumPy 的核心是 `ndarray`，要求同一数组元素类型相同（dtype）。金融计算默认用 `float64`。

In [ ]:
# ---- dtype 演示 ----
a_int   = np.array([1, 2, 3])                         # int64
a_float = np.array([10.0, 10.5, 10.3, 10.8])          # float64
a_f32   = np.array([1.0, 2.0], dtype=np.float32)      # float32，节省内存

print("int64  dtype:", a_int.dtype)
print("float64 dtype:", a_float.dtype)
print("float32 dtype:", a_f32.dtype)

# ---- axis 演示 ----
# 模拟 3 天 × 4 只股票的价格矩阵
mat = np.array([
    [10., 50., 100., 20.],   # day 1
    [11., 52.,  98., 21.],   # day 2
    [10., 51., 102., 19.],   # day 3
])

print("\n沿 axis=0（时间）求均值（每只股票均价）:", mat.mean(axis=0))
print("沿 axis=1（标的）求均值（每天截面均价):", mat.mean(axis=1))

## §2  向量化 vs for 循环：性能对比

向量化利用 C 底层运算，避免 Python 解释器开销，通常快 **50–200 倍**。

In [ ]:
import time

rng = np.random.default_rng(42)
n = 500_000
prices_big = rng.uniform(50, 150, size=n)

# 方式一：for 循环
t0 = time.perf_counter()
ret_loop = []
for i in range(1, len(prices_big)):
    ret_loop.append(prices_big[i] / prices_big[i - 1] - 1)
t_loop = time.perf_counter() - t0

# 方式二：向量化
t0 = time.perf_counter()
ret_vec = prices_big[1:] / prices_big[:-1] - 1
t_vec = time.perf_counter() - t0

print(f"for 循环耗时：{t_loop*1000:7.1f} ms")
print(f"向量化耗时：  {t_vec*1000:7.2f} ms")
print(f"加速比：      {t_loop/t_vec:.0f}x")
print(f"结果一致：    {np.allclose(ret_loop, ret_vec)}")

## §3  广播规则（Broadcasting）

广播让不同形状的数组可以一起运算，无需手动复制数据。

规则：从最后一维对齐，某维大小为 1 则自动扩展到匹配对方的大小。

In [ ]:
rng = np.random.default_rng(42)
mat = rng.standard_normal((250, 4))   # 250 天 × 4 只股票

# 1. 去截面均值：mat (250,4) - mean (4,) → 广播 mean 到 (250,4)
demeaned = mat - mat.mean(axis=0)
print("去均值后，每列均值 ≈ 0:", demeaned.mean(axis=0).round(10))

# 2. 标准化（Z-score）
z_score = (mat - mat.mean(axis=0)) / mat.std(axis=0)
print("Z-score 均值:", z_score.mean(axis=0).round(10))
print("Z-score 标准差:", z_score.std(axis=0).round(6))

# 3. 权重 × 收益矩阵 → 组合收益
w = np.array([0.4, 0.3, 0.2, 0.1])     # 权重
port_ret_broadcast = (mat * w).sum(axis=1)   # 广播
port_ret_matmul    = mat @ w                  # 矩阵乘（等价）
print("两种方式结果一致:", np.allclose(port_ret_broadcast, port_ret_matmul))

## §4  基础线性代数：矩阵乘 `@` 与组合收益

金融中大量公式可用矩阵乘表达：
- 组合预期收益：$\mu_p = w^\top \mu$
- 组合方差：$\sigma_p^2 = w^\top \Sigma w$

In [ ]:
# 预期年化收益与协方差（简化示例）
mu    = np.array([0.10, 0.12, 0.15, 0.08])       # 预期年化收益
w     = np.array([0.25, 0.25, 0.25, 0.25])        # 等权组合
Sigma = np.diag([0.04, 0.09, 0.16, 0.01])         # 对角协方差（无相关）

port_mu  = w @ mu            # w^T μ
port_var = w @ Sigma @ w     # w^T Σ w
port_vol = port_var ** 0.5

print(f"组合预期收益率：{port_mu:.2%}")
print(f"组合方差：      {port_var:.4f}")
print(f"组合波动率：    {port_vol:.2%}")

# 矩阵转置
A = np.array([[1, 2], [3, 4], [5, 6]])
print("\nA shape:", A.shape, "  A.T shape:", A.T.shape)
print("A.T @ A =\n", A.T @ A)

## §5  Pandas：加载内置数据与基本结构

In [ ]:
prices = load_sample_prices()

print("类型:", type(prices))
print("形状:", prices.shape)
print("索引类型:", type(prices.index))
print("日期范围:", prices.index[0].date(), "→", prices.index[-1].date())
print("列名:", prices.columns.tolist())
print("\n前 5 行:")
prices.head()

## §6  索引：loc / iloc / 布尔索引

In [ ]:
# loc：按标签（包含两端）
q1_2025 = prices.loc["2025-01":"2025-03"]
print(f"2025 Q1 交易日数：{len(q1_2025)}")

# loc：选行 + 选列
bank_2025 = prices.loc["2025", "BANK"]
print(f"BANK 2025年均价：{bank_2025.mean():.2f}")

# iloc：按整数位置（不含右端）
first3 = prices.iloc[:3, :2]    # 前3行、前2列
print("\nioc[0:3, 0:2]:")
print(first3)

# 布尔索引：TECH 超过历史均值的交易日
tech_mean = prices["TECH"].mean()
above_avg = prices[prices["TECH"] > tech_mean]
print(f"\nTECH 均值：{tech_mean:.2f}，超过均值天数：{len(above_avg)}")

## §7  缺失值处理 + groupby 按年统计

In [ ]:
# ---- 缺失值演示 ----
prices_copy = prices.copy()
# 人工制造缺失
prices_copy.iloc[5:10, 0] = np.nan

print("缺失数量:")
print(prices_copy.isna().sum())

filled = prices_copy.ffill()
print("\n前向填充后缺失数量:")
print(filled.isna().sum())

# ---- 按年分组统计日度收益率 ----
ret = daily_returns(prices)   # 算术日收益率

annual_stats = (
    ret.groupby(ret.index.year)
    .agg(["mean", "std"])
    .round(4)
)
annual_stats.index.name = "year"
print("\n各年日度收益率均值与标准差（前4列）:")
annual_stats.iloc[:, :4]

## §8  merge 与 concat：多标的数据对齐

In [ ]:
# concat：把两个错位的 Series 拼在一起
bank   = prices[["BANK"]].copy()
liquor = prices[["LIQUOR"]].iloc[5:].copy()   # 故意去掉前 5 行

merged = pd.concat([bank, liquor], axis=1)
print("拼合后形状:", merged.shape)
print("LIQUOR 前5行（应为 NaN）:")
print(merged[["LIQUOR"]].head())

# merge：月度价格与模拟利率数据按年月对齐
stock_m = prices.resample("ME").last().copy()
stock_m["ym"] = stock_m.index.to_period("M")

# 模拟利率数据
rate_df = pd.DataFrame({
    "ym":   pd.period_range(stock_m["ym"].iloc[0], periods=len(stock_m), freq="M"),
    "rate": np.linspace(3.5, 4.2, len(stock_m)),
})

combined = stock_m.merge(rate_df, on="ym", how="left")
print("\n月度价格 + 利率数据（前3行）:")
combined[["BANK", "TECH", "rate"]].head(3)

## §9  resample：日 → 月/周/年

> Pandas 2.x 频率别名：`ME`（月末）、`QE`（季末）、`YE`（年末），旧版 `M`/`Q`/`Y` 已弃用。

In [ ]:
# 月末价格
month_end = prices.resample("ME").last()
# 月度算术收益率
month_ret = month_end.pct_change().dropna()
print("月度收益率（前5行）:")
print(month_ret.head())

# 周频收益率（每周五）
week_ret = prices.resample("W-FRI").last().pct_change().dropna()
print(f"\n周频收益率条数：{len(week_ret)}")

# 年末价格
annual_price = prices.resample("YE").last()
print("\n年末价格:")
print(annual_price)

## §10  rolling 与 ewm：移动窗口统计

In [ ]:
tech = prices["TECH"]

# 20 日简单移动平均
sma20 = tech.rolling(window=20).mean()

# 20 日 EWMA（指数加权）
ewma20 = tech.ewm(span=20, adjust=False).mean()

# 60 日滚动波动率（年化）
vol60 = tech.pct_change().rolling(60).std() * (252 ** 0.5)

# EWM 波动率（RiskMetrics 常用）
ewm_vol = tech.pct_change().ewm(span=60).std() * (252 ** 0.5)

# 打印后 5 行对比
comparison = pd.DataFrame({
    "价格": tech,
    "SMA20": sma20,
    "EWMA20": ewma20,
    "滚动波动率(年化)": vol60,
    "EWM波动率(年化)": ewm_vol,
})
print(comparison.tail())

## §11  Parquet 读写与往返一致性

In [ ]:
import os, tempfile

tmp_dir = tempfile.gettempdir()
parquet_path = os.path.join(tmp_dir, "ch02_prices.parquet")
csv_path     = os.path.join(tmp_dir, "ch02_prices.csv")

# 写入
prices.to_parquet(parquet_path, compression="snappy")
prices.to_csv(csv_path, encoding="utf-8")

# 读回
back_parquet = pd.read_parquet(parquet_path)
back_csv     = pd.read_csv(csv_path, index_col=0, parse_dates=True)

print("Parquet 往返一致：", prices.equals(back_parquet))
print("CSV 往返一致（索引类型需额外处理）：", prices.values == back_csv.values, "\n")

# 文件大小对比
p_size = os.path.getsize(parquet_path)
c_size = os.path.getsize(csv_path)
print(f"Parquet 大小：{p_size:,} bytes")
print(f"CSV     大小：{c_size:,} bytes")
print(f"压缩比（CSV/Parquet）：{c_size/p_size:.1f}x")

## §12  matplotlib 可视化：含中文标注

In [ ]:
# 确保中文字体已设置（§0 已调用，这里演示图表效果）
tech = prices["TECH"]
sma20  = tech.rolling(20).mean()
ewma20 = tech.ewm(span=20, adjust=False).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# 上图：股价 + 均线
axes[0].plot(tech.index, tech, alpha=0.4, color="steelblue", linewidth=1, label="科技股价格")
axes[0].plot(sma20.index, sma20, color="orange", linewidth=2, label="20日SMA")
axes[0].plot(ewma20.index, ewma20, color="green", linewidth=2, linestyle="--", label="20日EWMA")
axes[0].set_ylabel("价格（元）")
axes[0].set_title("科技股移动平均对比")
axes[0].legend(loc="upper left")

# 下图：60日滚动年化波动率
vol60 = tech.pct_change().rolling(60).std() * (252**0.5)
axes[1].fill_between(vol60.index, vol60, alpha=0.3, color="tomato")
axes[1].plot(vol60.index, vol60, color="tomato", linewidth=1.2, label="60日滚动波动率（年化）")
axes[1].set_ylabel("波动率")
axes[1].set_xlabel("日期")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
axes[1].legend()

fig.tight_layout()
plt.show()
print("图表绘制完成（含中文标注）")

---
## 习题参考解答

以下各题均可独立运行。

In [ ]:
# ---- 习题1：向量化 5 日移动平均 ----
prices = load_sample_prices()
liquor = prices["LIQUOR"].values   # NumPy array

# NumPy convolve 实现等权 SMA
kernel = np.ones(5) / 5
sma_np = np.convolve(liquor, kernel, mode="valid")   # 长度 = n - 4

# Pandas rolling 实现（丢弃前 4 个 NaN 后对齐）
sma_pd = prices["LIQUOR"].rolling(5).mean().dropna().values

print("习题1：两种方法结果一致：", np.allclose(sma_np, sma_pd))
print(f"  NumPy  前3值：{sma_np[:3].round(4)}")
print(f"  Pandas 前3值：{sma_pd[:3].round(4)}")

In [ ]:
# ---- 习题2：广播计算超额收益 ----
ret = daily_returns(prices)                   # T × 4 DataFrame
ret_np = ret.values                           # NumPy array

# 每日等权平均（截面均值）
cross_mean = ret_np.mean(axis=1, keepdims=True)   # shape (T, 1)，keepdims 保证广播
excess = ret_np - cross_mean                       # 广播：(T,4) - (T,1)

# 转回 DataFrame
excess_df = pd.DataFrame(excess, index=ret.index, columns=ret.columns)
print("习题2：超额收益（前5行）:")
print(excess_df.head().round(5))
print("每日截面均值是否≈0:", np.allclose(excess_df.mean(axis=1), 0))

In [ ]:
# ---- 习题3：周频收益率 + 按年分组统计 ----
week_prices = prices.resample("W-FRI").last()
week_ret    = week_prices.pct_change().dropna()

annual_weekly = (
    week_ret
    .groupby(pd.Grouper(freq="YE"))
    .agg(["mean", "std"])
    .round(4)
)
annual_weekly.index = annual_weekly.index.year
annual_weekly.index.name = "年份"
print("习题3：各年周度收益率均值与标准差:")
print(annual_weekly)

# 找波动最大的年份（以 TECH 为例）
max_vol_year = annual_weekly[("TECH", "std")].idxmax()
print(f"\nTECH 周度波动最大年份：{max_vol_year}")

In [ ]:
# ---- 习题4：Parquet 往返 + 文件大小对比 ----
import os, tempfile

ret = daily_returns(prices)

# 找波动最高的股票
most_volatile = ret.std().idxmax()
print(f"波动最高股票：{most_volatile}")

series_to_save = ret[[most_volatile]]

tmp = tempfile.gettempdir()
pq = os.path.join(tmp, "ch02_ex4.parquet")
cs = os.path.join(tmp, "ch02_ex4.csv")

series_to_save.to_parquet(pq)
series_to_save.to_csv(cs)

back = pd.read_parquet(pq)
print(f"Parquet 往返一致：{series_to_save.equals(back)}")
print(f"Parquet 大小：{os.path.getsize(pq):,} bytes")
print(f"CSV     大小：{os.path.getsize(cs):,} bytes")

In [ ]:
# ---- 习题5：TECH 三条均线可视化 ----
set_chinese_font()

tech   = prices["TECH"]
sma20  = tech.rolling(20).mean()
ewma20 = tech.ewm(span=20, adjust=False).mean()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(tech.index, tech, alpha=0.4, linewidth=1,
        color="steelblue", label="科技股日收盘价")
ax.plot(sma20.index, sma20, linewidth=2,
        color="orange", label="20日SMA")
ax.plot(ewma20.index, ewma20, linewidth=2, linestyle="--",
        color="green", label="20日EWMA")

ax.set_title("科技股移动平均对比")
ax.set_ylabel("价格（元）")
ax.set_xlabel("日期")
ax.legend()
fig.tight_layout()

out_path = os.path.join(tempfile.gettempdir(), "ch02_ex5.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"图已保存至：{out_path}")